# NEMI on fronts

Clusters fronts rather than patches.  The cutout pipeline turns images into
vectors before it can cluster them; a front is already a row of numbers, so the
CNN trunk and the patch machinery drop out and `fronts_dataloader` hands NEMI a
matrix directly.

This notebook loads the small finding run — three SURF snapshots — and
assembles the feature matrix.  See `docs/fronts_dataloader.md` for the
selection and scaling contract.

## The store

In [1]:
import numpy as np
import pandas as pd

from fronts_dataloader.fronts_dataset import FrontDataSource, RawFronts

#: The small finding run, where build-fronts' push step publishes it.  The key
#: is assembled from the run config's source block, so the products sit beside
#: the stores they were derived from:
#:     {bucket}/{folder}/{run_id}/Fronts/{build_version}/{pipeline}/fronts.zarr
STORE = ("s3://dbof/globals_for_cutouts/v2_2_01"
         "/Fronts/SMALL_DATASET/SURF/fronts.zarr")

#: Nautilus, not AWS, so the endpoint has to be given explicitly.
S3_ENDPOINT = "https://s3-west.nrp-nautilus.io"

source = FrontDataSource(STORE, storage_options={"endpoint_url": S3_ENDPOINT})
print(f"{len(source.dates)} snapshots")
print(source.store.status().to_string(index=False))

3 snapshots
           date find group colocate
20111216_030000 done  done     done
20120104_110000 done  done     done
20120208_230000 done  done     done


What `colocate` recorded decides what the features mean: which
statistics exist at all, how wide a band each was measured over, and whether
land was dropped or propagated.

In [2]:
# What the colocate step recorded -- these decide what the features mean.
attrs = source.store.step_attrs(source.dates[0], "colocate")
for key in ("stats", "percentiles", "properties_dilation_radius",
            "properties_cross_front_radius", "dilate_only_to_front_pixels",
            "min_npix", "nan_policy"):
    print(f"  {key:30s} {attrs.get(key, '(absent)')}")

for group, columns in source.feature_groups().items():
    print(f"\n{group}  ({len(columns)} columns)")
    print("   ", ", ".join(columns[:8]), "..." if len(columns) > 8 else "")

  stats                          ['mean', 'std', 'min', 'max', 'skew']
  percentiles                    [25, 75, 90]
  properties_dilation_radius     2
  properties_cross_front_radius  0
  dilate_only_to_front_pixels    True
  min_npix                       1
  nan_policy                     omit

geometry  (12 columns)
    npix, centroid_lat, centroid_lon, length_km, orientation, num_branches, lat_min, lat_max ...

properties  (241 columns)
    npix, gradb2_mean, gradb2_std, gradb2_min, gradb2_max, gradb2_skew, gradb2_p25, gradb2_p75 ...


## Load

Every snapshot, one row per front.  Load once and `select` repeatedly — the load is the expensive half.

In [ ]:
raw = RawFronts.load(source)

print(f"{len(raw):,} fronts over {len(source.dates)} snapshots")
print(raw.table.groupby("date").size().rename("fronts").to_string())

## Select features

`select` takes groups (`geometry`, `properties`, `cross`), channels
(`gradb2`, which expands to each of its statistics), or single columns, mixed
in one list.

`nan_policy="fill"` keeps every front and every feature.  `fill_value="mean"`
puts an unmeasured front at its column's centre, which asserts nothing; the
`_missing` indicator columns keep a filled value distinguishable from a
measured one.

In [ ]:
# FEATURES = ["mean_curvature", "length_km", "orientation", "num_branches", "curvature_direction", ## NOTE these features create a map of abs of lat
#             "gradb2", "Theta", "Eta", "density", "gradeta2", "oceQnet", "okubo_weiss",
#             "rossby_number", "strain_mag", "strain_s", "strain_n", "wind_stress_curl"]
#
# ds = raw.select(FEATURES, stats=("mean", "std", "skew"), scaling="standardize", nan_policy="fill", div_by_f=True,
#                 fill_value="mean", missing_indicator=False)
# print(ds.summary())


#
# FEATURES = ["length_km", "num_branches"] #, "rossby_number", "strain_mag", "strain_s", "strain_n", "gradb2"] #            , "Eta", "gradeta2",orientation


# NO lat features
FEATURES = ["length_km", "num_branches", "orientation",
            "rossby_number","strain_mag", "strain_s", "strain_n", "gradb2"]


ds = raw.select(FEATURES, stats=("std", "skew"), scaling="standardize", nan_policy="fill", div_by_f=True,
                fill_value="mean", missing_indicator=False)
print(ds.summary())


# A column filled for most fronts describes the fill, not the ocean.
print("\nmost-filled columns:")
for column, info in ds.worst_filled()[:8]:
    print(f"  {column:30s} {info['n']:8,d}  {100 * info['n'] / len(ds):5.1f}%")

ds.summary()


`mean_curvature` is missing for 20%, and that is not "this front is straight".
The estimator needs a skeleton longer than `2 * window_size = 10` px, so every
short front comes back undefined.  Straight is measurable and comes out near
zero.

## The matrix

`X` is what `nemi.run` takes; `ids` are the same rows in the same order, so a cluster label joins back to the store on `(date, label)`.

In [ ]:
print("X", ds.X.shape, ds.X.dtype, " finite:", bool(np.isfinite(ds.X).all()))
ds.to_frame().head()

#ds.to_frame().shape

#NOTE the first 4 are not included in clustering

## Sweep

`run_sweep` takes the two NEMI dictionaries as grids and expands them
full-factorially.  Each embedding is fitted once and reused for every
clustering configuration on top of it, so the cost is the size of the embedding
grid, not the product of the two.

`fit_size` subsamples before fitting -- a full UMAP per cell is not what you
want while choosing parameters.

In [ ]:
from nemi_sweep.sweep import run_sweep
from visualization.sweep_plots import (plot_all_metric_heatmaps,
                                       plot_embedding_grid)

EMBEDDING_GRID = {"n_neighbors": [5, 50, 100, 150, 200], "min_dist": [0.0, 0.05]}
CLUSTERING_GRID = {"method": ["dbscan"], "eps": [0.05, 0.1, 0.5], "min_samples": [5, 10, 50, 100]}

result = run_sweep(ds.X, EMBEDDING_GRID, CLUSTERING_GRID,
                   n_components=3, device="cpu",
                   fit_size=20_000, metric_sample=3_000, keep_size=20_000)


In [ ]:
print(result.summary())

One row per (embedding x clustering) configuration, parameters beside metrics.  `normalized_stress` and `noise_%` read lower-is-better; the rest higher.

In [ ]:
result.table.sort_values("trustworthiness", ascending=False)

## Plot

Both put the same two swept parameters on the same axes, so a heatmap cell and the scatter beneath it are the same configuration.

In [ ]:
plot_all_metric_heatmaps(result, x="n_neighbors", y="min_dist")

In [ ]:
plot_all_metric_heatmaps(result, x="min_samples", y="eps")

In [ ]:
plot_embedding_grid(result, x="n_neighbors", y="min_dist", dims=3, alpha=0.5);

## Read the embedding

The sweep plots ask which configuration to keep.  These ask what the
kept one is organized by: same embedding, one panel per variable, so a
gradient across the cloud reads as a gradient.

The sweep stores a subsample of the rows it fitted, so anything used as
a color has to be indexed with `result.row_index` first — the plots
raise rather than silently misalign.

In [ ]:
from visualization.embedding_plots import (plot_embedding_by,
                                           plot_embedding_features)

best = result.table["trustworthiness"].idxmax()
E = result.embeddings[best]
print(result.table.loc[best, list(result.swept)].to_dict(), E.shape)

# centroid_lat was never selected as a feature, so it comes off the
# source table on X's rows, then down to the rows this embedding kept.
meta = ds.meta(raw.table, ["centroid_lat", "centroid_lon", 'SIarea_mean', 'gradb2_mean']).iloc[result.row_index]

plot_embedding_by(E, meta["centroid_lat"].to_numpy(), dims=3, name="centroid_lat")

In [ ]:
plot_embedding_by(E, labels, dims=3, name="centroid_lat")

In [ ]:
plot_embedding_by(E, meta['SIarea_mean'].to_numpy(), dims=2, name="ice")

In [ ]:
plot_embedding_by(E, meta['gradb2_mean'].to_numpy(), dims=2, name="grad B^2");

In [ ]:
plot_embedding_by(E, np.abs(meta["centroid_lat"].to_numpy()), name="|centroid_lat|", dims=3, alpha=0.3);

In [ ]:
ds.feature_names

A clean latitude gradient across the cloud means the embedding is a map
of where fronts are, not a taxonomy of what they are.  Most features
carry latitude — `Theta_mean` correlates with it at 0.95 — and
`div_by_f` only takes it out of the strain columns.

In [ ]:
PANELS = ["length_km", "gradb2_mean", "rossby_number_mean",
          "strain_mag_mean", "mean_curvature", "num_branches", "Theta_mean",
          "wind_stress_curl_mean"]

# Unscaled values, so each colorbar reads in the variable's own units.
frame = pd.DataFrame(ds.raw, columns=ds.feature_names)[PANELS].iloc[result.row_index]
plot_embedding_features(E, frame, n_cols=3);